In [1]:
import pandas

/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df =pd.read_csv('Master_Polling_Data.csv')

<IPython.core.display.Javascript object>

In [3]:
df.columns

Index(['Serial No. Of Polling Station', 'Naam Tamilar Katchi',
       'Dravida Munnetra Kazhagam',
       'Nam Naadu Nam Makkal Nam Ethirkaalam Katchi', 'Puthiya Tamilagam',
       'Amma Makkal Munnettra Kazagam', 'Naam Indiar Party',
       'Tamizhaga Vaazhvurimai Katchi', 'Tamilaga Vettri Kazhagam',
       'Independent', 'Independent.1', 'Independent.2', 'Independent.3',
       'Independent.4', 'Total of Valid Votes', 'Unnamed: 15', 'NOTA', 'Total',
       'No. Of Tendered Votes', 'Locality',
       'Building  in  Which  it  will  be  Located', 'Polling  Area',
       'Whether  for  all Voters  or  men  olny or  women  only',
       'Naam Tamilar Katchi_Share_%', 'Dravida Munnetra Kazhagam_Share_%',
       'Nam Naadu Nam Makkal Nam Ethirkaalam Katchi_Share_%',
       'Puthiya Tamilagam_Share_%', 'Amma Makkal Munnettra Kazagam_Share_%',
       'Naam Indiar Party_Share_%', 'Tamizhaga Vaazhvurimai Katchi_Share_%',
       'Tamilaga Vettri Kazhagam_Share_%', 'NOTA_Share_%',
       'Total_

In [5]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 1. Load the new dataset (Make sure to update the filename)
df2 = pd.read_csv("Master_Polling_Data.csv")

# 2. Select the key political parties based on your new column index
new_party_shares = [
    'Dravida Munnetra Kazhagam_Share_%',
    'Tamilaga Vettri Kazhagam_Share_%',
    'Naam Tamilar Katchi_Share_%',
    'Puthiya Tamilagam_Share_%',
    'Amma Makkal Munnettra Kazagam_Share_%'
]

# 3. Handle any missing data in the share columns
df2[new_party_shares] = df2[new_party_shares].fillna(0)

# 4. Include structural features for voter behavior analysis
# We use Independent_Share_% and Margin_Percentage to track competition intensity
feature_cols = new_party_shares + ['Independent_Share_%', 'Margin_Percentage']
df2[feature_cols] = df2[feature_cols].fillna(0)

# 5. Extract and scale the features
X = df2[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df2['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Profile Breakdown of each voter group
print("\n--- NEW DATASET: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df2.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- NEW DATASET: BOOTH COUNT PER CLUSTER ---")
print(df2['Cluster_ID'].value_counts())

# 8. Automatically export separate action lists for campaign ground teams
for cluster_num in range(optimal_k):
    cluster_df = df2[df2['Cluster_ID'] == cluster_num][
        ['Serial No. Of Polling Station', 'Locality', 'Building  in  Which  it  will  be  Located', 'Polling  Area', 'Winner_Party', 'Margin_Percentage']
    ]
    # Save each list to a separate file
    filename = f"New_Dataset_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")



--- NEW DATASET: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            Dravida Munnetra Kazhagam_Share_%  \
Cluster_ID                                      
0                                       35.10   
1                                       23.45   
2                                       24.16   
3                                       19.75   

            Tamilaga Vettri Kazhagam_Share_%  Naam Tamilar Katchi_Share_%  \
Cluster_ID                                                                  
0                                      38.66                         9.11   
1                                      54.97                         9.66   
2                                      36.06                         7.78   
3                                      25.49                         6.74   

            Puthiya Tamilagam_Share_%  Amma Makkal Munnettra Kazagam_Share_%  \
Cluster_ID                                                                     
0                  

In [6]:
import pandas as pd

# 1. Load your second dataset (ensure the filename matches your file)
df2 = pd.read_csv("Master_Polling_Data.csv")

# 2. Map the Cluster numbers to their actual political meaning
cluster_map = {
    0: "TVK vs DMK Battleground",
    1: "TVK Landslide Wave",
    2: "AMMK Spoiler Zone",
    3: "Puthiya Tamilagam Base"
}

# 3. Apply the human-readable names to the dataset
df2['Political_Profile_Name'] = df2['Cluster_ID'].map(cluster_map)

# 4. Save the global master file with the new descriptive text column
df2.to_csv("Constituency_2_With_Party_Labels.csv", index=False)

# 5. Automatically generate updated individual files for your teams
for cluster_num, party_label in cluster_map.items():
    # Filter the booths belonging to the current cluster
    filtered_df = df2[df2['Cluster_ID'] == cluster_num][
        ['Serial No. Of Polling Station', 'Locality', 'Building  in  Which  it  will  be  Located', 'Polling  Area', 'Winner_Party', 'Margin_Percentage', 'Political_Profile_Name']
    ]
    
    # Create a clean, descriptive filename using the party/profile name
    clean_filename = f"Booths_{party_label.replace(' ', '_')}.csv"
    filtered_df.to_csv(clean_filename, index=False)

print("Success! Created descriptive files:")
for label in cluster_map.values():
    print(f"- Booths_{label.replace(' ', '_')}.csv")


KeyError: 'Cluster_ID'